In [ ]:
import pandas as pd
from dotenv import load_dotenv
import os
import matplotlib

load_dotenv()

True

## Chargement données de consommation électrique en France entre 2020 et 2025

In [86]:
data_storage = os.getenv("DATA_STORAGE")

df_rte = pd.read_csv(data_storage + '/power_consumption_2000_2025.csv')
df_rte.shape

/tmp/ipykernel_7598/3018419712.py:3: DtypeWarning:

Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.



(447024, 5)

In [87]:
# Suppression colonnes inutiles
df_rte.drop(['status','updated_date'], axis=1, inplace=True)
print(df_rte.shape)
df_rte.head()

(447024, 3)


,start_date,end_date,value
0,2000-01-01T00:00:00+01:00,2000-01-01T00:30:00+01:00,50732
1,2000-01-01T00:30:00+01:00,2000-01-01T01:00:00+01:00,49456
2,2000-01-01T01:00:00+01:00,2000-01-01T01:30:00+01:00,49924
3,2000-01-01T01:30:00+01:00,2000-01-01T02:00:00+01:00,50018
4,2000-01-01T02:00:00+01:00,2000-01-01T02:30:00+01:00,49493


In [88]:
print(df_rte.describe())
print(df_rte.info())

               value
count  447024.000000
mean    53499.492392
std     11471.471916
min     28545.000000
25%     44761.000000
50%     52102.000000
75%     61059.000000
max    102098.000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 447024 entries, 0 to 447023
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   start_date  447024 non-null  object
 1   end_date    447024 non-null  object
 2   value       447024 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 10.2+ MB
None


In [89]:
# conversion champs dates necessite un passage en utc pour gérer les timezones différents du au changement d'horaire
# ajout colonne année
df_rte['start_date'] = pd.to_datetime(df_rte['start_date'], utc=True).dt.tz_convert('Europe/Paris')
df_rte['end_date'] = pd.to_datetime(df_rte['end_date'], utc=True).dt.tz_convert('Europe/Paris')
df_rte['year'] = df_rte['start_date'].dt.year
df_rte['month'] = df_rte['start_date'].dt.month
df_rte['week'] = df_rte['start_date'].dt.isocalendar().week
df_rte['day_of_week'] = df_rte['start_date'].dt.dayofweek
df_rte['day_of_year'] = df_rte['start_date'].dt.dayofyear
df_rte['period'] = pd.cut(df_rte['year'], [1999,2004,2009,2014,2019,2025], labels=['2000-2004', '2005-2009', '2010-2014', '2015-2019', '2020-2025'])
df_rte['day_of_year_align'] = (df_rte['week'] - 1) * 7 + df_rte['day_of_week']
print(df_rte.dtypes)
df_rte.sample(10)

start_date           datetime64[ns, Europe/Paris]
end_date             datetime64[ns, Europe/Paris]
value                                       int64
year                                        int32
month                                       int32
week                                       UInt32
day_of_week                                 int32
day_of_year                                 int32
period                                   category
day_of_year_align                           Int64
dtype: object


,start_date,end_date,value,year,month,week,day_of_week,day_of_year,period,day_of_year_align
82770,2004-09-20 09:00:00+02:00,2004-09-20 09:30:00+02:00,53966,2004,9,39,0,264,2000-2004,266
138258,2007-11-20 09:00:00+01:00,2007-11-20 09:30:00+01:00,72708,2007,11,47,1,324,2005-2009,323
376737,2021-11-27 16:30:00+01:00,2021-11-27 17:00:00+01:00,65756,2021,11,47,5,331,2020-2025,327
342893,2019-07-23 14:30:00+02:00,2019-07-23 15:00:00+02:00,57128,2019,7,30,1,204,2015-2019,204
171494,2009-10-12 19:00:00+02:00,2009-10-12 19:30:00+02:00,58298,2009,10,42,0,285,2005-2009,287
383141,2022-04-10 02:30:00+02:00,2022-04-10 03:00:00+02:00,52114,2022,4,14,6,100,2020-2025,97
116983,2006-09-03 03:30:00+02:00,2006-09-03 04:00:00+02:00,34090,2006,9,35,6,246,2005-2009,244
296782,2016-12-04 23:00:00+01:00,2016-12-04 23:30:00+01:00,68192,2016,12,48,6,339,2015-2019,335
22780,2001-04-19 14:00:00+02:00,2001-04-19 14:30:00+02:00,61168,2001,4,16,3,109,2000-2004,108
15876,2000-11-26 18:00:00+01:00,2000-11-26 18:30:00+01:00,51376,2000,11,47,6,331,2000-2004,328


In [39]:
df_count_by_year = df_rte.groupby('year').count()

df_count_by_year.T

year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
start_date,17568,17520,17520,17520,17568,17520,17520,17520,17568,17520,...,17568,17520,17520,17520,10224,17520,17520,17520,17568,16032
end_date,17568,17520,17520,17520,17568,17520,17520,17520,17568,17520,...,17568,17520,17520,17520,10224,17520,17520,17520,17568,16032
value,17568,17520,17520,17520,17568,17520,17520,17520,17568,17520,...,17568,17520,17520,17520,10224,17520,17520,17520,17568,16032


In [64]:
# comparaison consommation moy par mois
df_year_by_month = df_rte[['value', 'year', 'month','period']].groupby(['month', 'year', 'period'])['value'].mean().reset_index()

fig1 = px.line(df_year_by_month, x="month", y="value", color="year", facet_row="period", 
    title="Consommation moyenne par mois", width=1000, height=1000)
fig1.update_xaxes(tickvals=list(range(1,13)))
fig1.show()


/tmp/ipykernel_7598/2933726867.py:2: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [92]:
df2 = df_rte[ (df_rte['year'].isin([2021,2022,2023])) & (df_rte['month'] == 3)]

df2_grouped = df2.groupby(['day_of_year_align', 'year'])['value'].mean().reset_index()

fig = px.line(df2_grouped, x='day_of_year_align', y="value", color = 'year', title="comparaison consommation sur un mois de mars")
fig.show()


In [97]:
df3 = df_rte[ (df_rte['year'] == 2021) & (df_rte['week'] == 10) ]

fig = px.line(df3, x='start_date', y="value", color="year", title="comparaison consommation sur une semaine")
fig.show()